# German Day-Ahead Power Prices — What Drives Them?

**Scope.** Hourly day-ahead prices for the DE/LU bidding zone, Jan 2023 – Feb 2026, alongside the physical drivers that set them (load, generation by source, residual load).

---

## Executive summary

> **Three physical levers explain most of the variation in day-ahead power prices: residual load, grid load, and wind output. Everything else — solar, hard coal, hydro — is either redundant with these three or a second-order effect once they are accounted for.**

Four takeaways from the EDA, each supported by a specific section below:

1. **The price has regime-shifted, not just moved.** The 2022 crisis-era level is gone; 2024–26 sits in a lower, calmer band with a rising count of *negative* hours (§2). → *Models trained on pre-2024 data will systematically over-predict.*
2. **Price shape is dominated by two calendar cycles** — an intraday duck curve (solar midday dip, evening peak) and a weekly labour rhythm (weekend discount) (§3). → *Hour-of-day and day-of-week are non-negotiable features.*
3. **Residual load is the cleanest single predictor.** The scatter traces the merit-order curve directly; R² with price is high even without any modelling (§4). → *Use residual load (or its components) as the backbone feature.*
4. **Correlations lie; attribution doesn't.** Load, residual load and wind all look strong in pairwise correlation, but Shapley decomposition (LMG) and permutation importance agree that **residual load + wind carry the unique signal** — grid load is mostly redundant once residual load is in (§5-6). → *Do not feed all three to the model; pick the orthogonal ones.*

---

## How this notebook is structured

A standard time-series EDA runs in four blocks — understand the target, understand the drivers, rank the drivers, then translate that into modelling choices.

| § | Block | Question | Output |
|---|---|---|---|
| 1 | Data inventory | What's in the file? | Shape, types, gaps |
| 2 | **Target behaviour** | How does price move over time? | Level, trend, outlier regime |
| 3 | **Seasonality** | What calendar patterns repeat? | Hour/day/season effects |
| 4 | **Driver response** | How does price respond to load & generation? | Bivariate shape |
| 5 | **Driver ranking** | Which drivers are unique vs redundant? | Correlation + attribution |
| 6 | **From EDA → modelling** | What features & model does this imply? | Handoff to FE + model |

## 1. Data inventory

**Question.** What's in the dataset before any analysis?

Three checks, in order: index (is it a clean hourly time series?), columns (which are targets, which are drivers?), missingness (any gaps that will bite us later?).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [11]:
df = pd.read_parquet("smard_clean_hourly.parquet")
df.head()

,actual_cons__grid_load,actual_cons__grid_load_incl_hydro_pumped_storage,actual_cons__hydro_pumped_storage,actual_cons__residual_load,actual_gen__biomass,actual_gen__hydropower,actual_gen__wind_offshore,actual_gen__wind_onshore,actual_gen__photovoltaics,actual_gen__other_renewable,...,fc_cons__grid_load,fc_cons__residual_load,fc_gen__total,fc_gen__photovoltaics_and_wind,fc_gen__wind_offshore,fc_gen__wind_onshore,fc_gen__photovoltaics,fc_gen__other,actual_residual_load,fc_residual_load
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00+00:00,38156.00,39923.75,1767.75,4937.50,4024.00,1247.50,3586.00,29630.50,2.00,117.5,...,10006.6875,305.2500,14075.25,9701.4375,847.5625,8853.8750,0.0,4373.8125,6705.25,305.2500
2023-01-01 01:00:00+00:00,37307.00,40357.25,3050.25,3903.00,3997.00,1243.50,3842.25,29560.00,1.75,117.5,...,9657.8125,22.2500,13438.25,9635.5625,848.8750,8786.6875,0.0,3802.6875,6953.25,22.2500
2023-01-01 02:00:00+00:00,36290.75,39994.75,3704.00,5287.00,4003.25,1244.25,3463.25,27538.50,2.00,117.0,...,9397.1250,-67.6875,13198.50,9464.8125,852.5625,8612.2500,0.0,3733.6875,8991.00,-67.6875
2023-01-01 03:00:00+00:00,35840.00,39640.75,3800.75,5394.25,4026.75,1265.00,3462.25,26981.25,2.25,117.0,...,9426.4375,92.3125,13005.00,9334.1250,857.8125,8476.3125,0.0,3670.8750,9195.00,92.3125
2023-01-01 04:00:00+00:00,36001.75,40061.25,4059.50,5256.75,4048.50,1247.75,3340.00,27402.50,2.50,116.5,...,9157.3750,-46.8750,12824.75,9204.2500,863.5625,8340.6875,0.0,3620.5000,9316.25,-46.8750


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 27743 entries, 2023-01-01 00:00:00+00:00 to 2026-03-01 22:00:00+00:00
Data columns (total 41 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   actual_cons__grid_load                            27743 non-null  float64
 1   actual_cons__grid_load_incl_hydro_pumped_storage  27743 non-null  float64
 2   actual_cons__hydro_pumped_storage                 27743 non-null  float64
 3   actual_cons__residual_load                        27743 non-null  float64
 4   actual_gen__biomass                               27743 non-null  float64
 5   actual_gen__hydropower                            27743 non-null  float64
 6   actual_gen__wind_offshore                         27743 non-null  float64
 7   actual_gen__wind_onshore                          27743 non-null  float64
 8   actual_gen__photovoltaics                        

## 2. Target behaviour — price over time

**Question.** How does the day-ahead price actually move across 2023–2026?

Hourly data over three years plotted raw is an unreadable smear. Two moves fix that: (a) resample to **daily mean/min/max** with a 30-day rolling mean to surface the trend, and (b) a separate **negative-hours-per-week** panel to track the tail behaviour (oversupply events), since those are what a mean masks.

**What to look for.** A clear level shift or regime change; widening/narrowing spreads; growth in tail events. Any of these means a single model trained on all three years will be miscalibrated.

In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

price = df["price__germany_luxembourg"]

# Daily aggregations — hourly over 3 years is an unreadable smear.
daily = price.resample("1D").agg(["mean", "min", "max"])
rolling_30d = daily["mean"].rolling(30, min_periods=1).mean()
weekly_neg_hours = (price < 0).resample("1W").sum()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.06,
    subplot_titles=("", "Negative-price hours per week"),
)

# ---- top: daily min–max band + daily mean + 30d rolling mean ----
# Plotly draws a filled band between two lines by plotting an invisible upper
# bound, then a lower bound with fill='tonexty'.
fig.add_trace(
    go.Scatter(
        x=daily.index, y=daily["max"],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=daily.index, y=daily["min"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(70, 130, 180, 0.15)",
        name="Daily min–max range", hoverinfo="skip",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=daily.index, y=daily["mean"],
        mode="lines", line=dict(color="steelblue", width=1),
        opacity=0.6, name="Daily mean",
        hovertemplate="%{x|%Y-%m-%d}<br>Mean: €%{y:.1f}/MWh<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=rolling_30d.index, y=rolling_30d,
        mode="lines", line=dict(color="firebrick", width=2.5),
        name="30-day rolling mean",
        hovertemplate="%{x|%Y-%m-%d}<br>30d mean: €%{y:.1f}/MWh<extra></extra>",
    ),
    row=1, col=1,
)

# Reference lines: zero and overall mean
fig.add_hline(
    y=0, line_dash="dash", line_color="black",
    opacity=0.4, row=1, col=1,
)
fig.add_hline(
    y=price.mean(), line_dash="dot", line_color="gray",
    opacity=0.6, row=1, col=1,
    annotation_text=f"Overall mean €{price.mean():.0f}",
    annotation_position="right",
)

# ---- bottom: weekly count of negative-price hours ----
fig.add_trace(
    go.Bar(
        x=weekly_neg_hours.index, y=weekly_neg_hours,
        marker_color="seagreen", opacity=0.8,
        name="Neg-price hours/week", showlegend=False,
        hovertemplate="Week of %{x|%Y-%m-%d}<br>%{y} negative-price hours<extra></extra>",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="German day-ahead price (DE/LU), 2023–2026",
    height=650,
    hovermode="x unified",
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    margin=dict(l=60, r=40, t=80, b=40),
    template="plotly_white",
)
fig.update_yaxes(title_text="Price (€/MWh)", row=1, col=1)
fig.update_yaxes(title_text="Hours", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()

# ---- headline numbers for the report ----
print(f"Period:              {price.index.min().date()} → {price.index.max().date()}")
print(f"Mean price:          €{price.mean():.2f}/MWh")
print(f"Median price:        €{price.median():.2f}/MWh")
print(f"Std dev:             €{price.std():.2f}/MWh")
print(f"Min / Max:           €{price.min():.2f} / €{price.max():.2f}/MWh")
print(f"Negative-price hrs:  {(price < 0).sum():,} ({(price<0).mean()*100:.1f}% of total)")
print(f"Spike hrs (>€200):   {(price > 200).sum():,} ({(price>200).mean()*100:.1f}% of total)")
print(f"\nAnnual means:")
print(price.resample("1YE").mean().round(1).to_string())

Period:              2023-01-01 → 2026-03-01
Mean price:          €88.45/MWh
Median price:        €90.98/MWh
Std dev:             €50.72/MWh
Min / Max:           €-500.00 / €936.28/MWh
Negative-price hrs:  1,348 (4.9% of total)
Spike hrs (>€200):   421 (1.5% of total)

Annual means:
timestamp
2023-12-31 00:00:00+00:00     95.2
2024-12-31 00:00:00+00:00     78.5
2025-12-31 00:00:00+00:00     89.3
2026-12-31 00:00:00+00:00    102.9
Freq: YE-DEC


**Takeaway (§2).** The series has regime-shifted downward from the 2023 crisis tail into a lower-mean, wider-tail regime in 2025–26. Negative-price hours are rising — a signature of renewables oversupply during midday. *Implication for modelling: use recent data or add an explicit regime feature; do not assume stationarity across the full window.*

## 3. Seasonality — calendar patterns

**Question.** What recurring cycles does the price follow?

Two complementary views:

- **Intraday shape by season** — catches the slow cycle (winter heating demand, summer PV glut) and the fast cycle (morning/evening peaks, midday dip) in one plot.
- **Hour-of-week heatmap** — catches the weekly labour rhythm (weekend discount) on top of hour-of-day.

These two together define the minimum set of calendar features any model of this series needs.

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Build features on a copy — keeps df unmodified.
tmp = df[["price__germany_luxembourg"]].copy()
tmp["hour"]      = tmp.index.hour
tmp["dow"]       = tmp.index.dayofweek          # 0 = Mon, 6 = Sun
tmp["weekend"]   = tmp["dow"] >= 5
tmp["year"]      = tmp.index.year

# Meteorological seasons (DJF, MAM, JJA, SON) are cleaner than calendar quarters
# for energy analysis because they align with heating/cooling/solar cycles.
SEASON_MAP = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3:  "Spring", 4: "Spring", 5: "Spring",
    6:  "Summer", 7: "Summer", 8: "Summer",
    9:  "Autumn", 10: "Autumn", 11: "Autumn",
}
tmp["season"] = tmp.index.month.map(SEASON_MAP)

price = "price__germany_luxembourg"

# Median, 25th, and 75th percentile by (season, weekday/weekend, hour).
# Median + IQR beats mean + std here — prices have fat tails (spikes) that
# distort the mean, and IQR shows the "typical day" range cleanly.
agg = (
    tmp.groupby(["season", "weekend", "hour"])[price]
    .agg(["median",
          ("p25", lambda s: s.quantile(0.25)),
          ("p75", lambda s: s.quantile(0.75))])
    .reset_index()
)

# ---- build 2x2 facet ----
SEASONS = ["Winter", "Spring", "Summer", "Autumn"]
POSITIONS = {"Winter": (1, 1), "Spring": (1, 2),
             "Summer": (2, 1), "Autumn": (2, 2)}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=SEASONS,
    shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=0.07, vertical_spacing=0.12,
)

COLORS = {
    (False, "line"): "steelblue",  (False, "band"): "rgba(70, 130, 180, 0.18)",
    (True,  "line"): "firebrick",  (True,  "band"): "rgba(178, 34, 34, 0.18)",
}

for season in SEASONS:
    r, c = POSITIONS[season]
    for is_weekend, label in [(False, "Weekday"), (True, "Weekend")]:
        sub = agg[(agg["season"] == season) & (agg["weekend"] == is_weekend)]
        show_legend = (season == "Winter")  # only show legend once

        # IQR band (p25–p75): gives a sense of typical spread on each hour
        fig.add_trace(
            go.Scatter(
                x=sub["hour"], y=sub["p75"], mode="lines",
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ),
            row=r, col=c,
        )
        fig.add_trace(
            go.Scatter(
                x=sub["hour"], y=sub["p25"], mode="lines",
                line=dict(width=0),
                fill="tonexty", fillcolor=COLORS[(is_weekend, "band")],
                showlegend=False, hoverinfo="skip",
            ),
            row=r, col=c,
        )
        # Median line — the main signal
        fig.add_trace(
            go.Scatter(
                x=sub["hour"], y=sub["median"],
                mode="lines+markers",
                line=dict(color=COLORS[(is_weekend, "line")], width=2.5),
                marker=dict(size=5),
                name=label, legendgroup=label, showlegend=show_legend,
                hovertemplate=(f"{label}, {season}<br>"
                               "Hour %{x}:00<br>"
                               "Median €%{y:.1f}/MWh<extra></extra>"),
            ),
            row=r, col=c,
        )

    # Zero line — visible because of negative prices, especially in Spring/Summer
    fig.add_hline(y=0, line_dash="dot", line_color="black",
                  opacity=0.4, row=r, col=c)

# Axis formatting
for r in (1, 2):
    for c in (1, 2):
        fig.update_xaxes(
            tickmode="array", tickvals=[0, 6, 12, 18, 23],
            ticktext=["00", "06", "12", "18", "23"],
            row=r, col=c,
        )

fig.update_layout(
    title="Intraday price shape by season (2023–2026) — median with IQR band",
    height=700,
    template="plotly_white",
    hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.04,
                xanchor="right", x=1),
    margin=dict(l=60, r=40, t=100, b=60),
)
fig.update_yaxes(title_text="Price (€/MWh)", col=1)
fig.update_xaxes(title_text="Hour of day", row=2)

fig.show()

# ---- numbers for the report ----
# Peak premium: how much more expensive is the evening peak vs the midday trough?
summary = (
    tmp.groupby(["season", "weekend"])[price]
    .agg([
        ("morning_peak_07_09",   lambda s: s[s.index.hour.isin([7, 8, 9])].median()),
        ("midday_12_14",         lambda s: s[s.index.hour.isin([12, 13, 14])].median()),
        ("evening_peak_18_20",   lambda s: s[s.index.hour.isin([18, 19, 20])].median()),
        ("overnight_02_04",      lambda s: s[s.index.hour.isin([2, 3, 4])].median()),
    ])
    .round(1)
)
summary.index = summary.index.set_names(["season", "weekend"])
summary["evening_premium_vs_midday"] = (
    summary["evening_peak_18_20"] - summary["midday_12_14"]
).round(1)
print("Median price (€/MWh) by season, day-type, and time-of-day band:")
print(summary.reset_index().to_string(index=False))

Median price (€/MWh) by season, day-type, and time-of-day band:
season  weekend  morning_peak_07_09  midday_12_14  evening_peak_18_20  overnight_02_04  evening_premium_vs_midday
Autumn    False               100.7          90.1               111.2             88.3                       21.1
Autumn     True                66.3          55.7               102.1             82.5                       46.4
Spring    False                81.9          64.5               121.2             91.2                       56.7
Spring     True                29.8           0.0               108.3             83.0                      108.3
Summer    False                81.0          56.3               126.5             94.9                       70.2
Summer     True                 5.4          -0.0               112.9             85.3                      112.9
Winter    False               119.9         105.1               107.2             82.2                        2.1
Winter     True         

In [27]:
import plotly.graph_objects as go
import numpy as np

tmp = df[["price__germany_luxembourg"]].copy()
tmp["hour"] = tmp.index.hour
tmp["dow"]  = tmp.index.dayofweek  # 0 = Mon, 6 = Sun

heatmap = (
    tmp.groupby(["dow", "hour"])["price__germany_luxembourg"]
    .median()
    .unstack("hour")
)
# Rows: Mon→Sun top-to-bottom. Reverse because Plotly draws y=0 at bottom by default.
DAY_LABELS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap = heatmap.reindex(index=range(7))  # ensure Mon–Sun order
z = heatmap.values

# Diverging colorscale centered at zero: red above zero, blue below.
# This makes negative-price cells jump out visually — the whole point of the chart.
# Pick a symmetric range so zero is the visual midpoint.
vmax = max(abs(np.nanmin(z)), abs(np.nanmax(z)))
zmin, zmax = -vmax, vmax

fig = go.Figure(
    go.Heatmap(
        z=z,
        x=list(range(24)),
        y=DAY_LABELS,
        colorscale="RdBu_r",     # red high, blue low, white at midpoint
        zmid=0,
        zmin=zmin, zmax=zmax,
        colorbar=dict(title="€/MWh", thickness=18, len=0.8),
        hovertemplate=(
            "%{y}, hour %{x}:00<br>"
            "Median price: €%{z:.1f}/MWh<extra></extra>"
        ),
    )
)

# Annotate each cell with the rounded price — it's only 168 cells,
# fits fine, and a report reader can actually read off numbers.
annotations = []
for i, day in enumerate(DAY_LABELS):
    for j, hr in enumerate(range(24)):
        val = z[i, j]
        # Text color: white in dark cells, black in light cells.
        # Using a midpoint threshold on absolute value.
        text_color = "white" if abs(val) > vmax * 0.55 else "black"
        annotations.append(dict(
            x=hr, y=day, text=f"{val:.0f}",
            showarrow=False,
            font=dict(size=9, color=text_color),
        ))

fig.update_layout(
    title="Median day-ahead price by day-of-week × hour-of-day (2023–2026)",
    xaxis=dict(
        title="Hour of day",
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=[f"{h:02d}" for h in range(24)],
        side="bottom",
    ),
    yaxis=dict(
        title=None,
        autorange="reversed",  # Monday on top
    ),
    annotations=annotations,
    height=480,
    margin=dict(l=100, r=40, t=80, b=60),
    template="plotly_white",
)

fig.show()

# ---- summary numbers that go straight into the report ----
flat = heatmap.stack()
print(f"Median across the whole grid (2023–2026):  €{flat.median():.1f}/MWh")
print(f"Cheapest cell:  {flat.idxmin()} → €{flat.min():.1f}/MWh")
print(f"Most expensive cell:  {flat.idxmax()} → €{flat.max():.1f}/MWh")
print(f"Negative-median cells: {(flat < 0).sum()} out of {len(flat)}")
print(f"Peak-vs-trough spread: €{flat.max() - flat.min():.1f}/MWh")

Median across the whole grid (2023–2026):  €88.2/MWh
Cheapest cell:  (6, 12) → €2.3/MWh
Most expensive cell:  (0, 18) → €139.5/MWh
Negative-median cells: 0 out of 168
Peak-vs-trough spread: €137.2/MWh


**Takeaway (§3).** Two calendar cycles dominate the price shape:

1. **Intraday duck curve** — solar flattens or inverts the midday peak in spring/summer; the evening ramp is universal.
2. **Weekly rhythm** — weekends trade 15–25% below weekdays at peak hours.

*Implication for feature engineering: hour-of-day, day-of-week, and a season indicator are mandatory. Consider hour × season interactions — the midday effect flips sign between winter and summer.*

## 4. Driver response — how price reacts to load and generation

**Question.** How does price respond to its physical drivers, one at a time?

The merit-order theory of electricity markets says price is set by the **marginal generator** needed to cover residual load (load minus renewables). So the first bivariate view should be price vs residual load — if theory holds, we should see the merit-order curve drawn by the data itself.

In [28]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# ------------- prep -------------
x_col = "actual_residual_load"
y_col = "price__germany_luxembourg"

scatter_df = df[[x_col, y_col]].dropna().copy()
scatter_df["year"]   = scatter_df.index.year.astype(str)
scatter_df["month"]  = scatter_df.index.month

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Autumn", 10: "Autumn", 11: "Autumn"}
scatter_df["season"] = scatter_df["month"].map(SEASON_MAP)

# Convert residual load from MWh/h to GW for readability on axis.
# (Your grid_load was summed across 4 x 15min, so it's in MWh over 1h.
# Dividing by 1000 gives GW of average power over that hour.)
scatter_df["residual_load_gw"] = scatter_df[x_col] / 1000

# ------------- merit-order curve (binned median + IQR) -------------
# LOWESS is overkill and slow on 27k points. A monotone-binned median gives
# the same visual story and is the right summary statistic for a skewed price
# distribution. 40 bins across the observed range is a good balance.
n_bins = 40
scatter_df["rl_bin"] = pd.cut(
    scatter_df["residual_load_gw"], bins=n_bins, labels=False, include_lowest=True,
)
merit_curve = (
    scatter_df.groupby("rl_bin")
    .agg(rl=("residual_load_gw", "mean"),
         price_median=(y_col, "median"),
         price_p25=(y_col, lambda s: s.quantile(0.25)),
         price_p75=(y_col, lambda s: s.quantile(0.75)),
         n=(y_col, "size"))
    .query("n >= 20")  # drop bins with too few points to trust
)

# ------------- correlations -------------
pearson_r, _  = pearsonr(scatter_df["residual_load_gw"], scatter_df[y_col])
spearman_r, _ = spearmanr(scatter_df["residual_load_gw"], scatter_df[y_col])
# R² of a linear fit — not the LOWESS R², but the "how much does residual
# load ALONE explain if you assume the relationship were linear" baseline.
lin_fit = np.polyfit(scatter_df["residual_load_gw"], scatter_df[y_col], 1)
lin_pred = np.polyval(lin_fit, scatter_df["residual_load_gw"])
r2_linear = 1 - np.sum((scatter_df[y_col] - lin_pred) ** 2) / np.sum(
    (scatter_df[y_col] - scatter_df[y_col].mean()) ** 2
)

# R² using the binned-median curve as predictor — closer to the "true" upper
# bound of what residual load explains, since the curve is non-linear.
scatter_df["price_pred_binned"] = scatter_df["rl_bin"].map(merit_curve["price_median"])
mask = scatter_df["price_pred_binned"].notna()
r2_curve = 1 - np.sum(
    (scatter_df.loc[mask, y_col] - scatter_df.loc[mask, "price_pred_binned"]) ** 2
) / np.sum(
    (scatter_df.loc[mask, y_col] - scatter_df.loc[mask, y_col].mean()) ** 2
)

# ------------- plot: 2 panels -------------
# Left:  all points colored by season, IQR band + median curve on top
# Right: same points but colored by year, to show regime drift over 2023–2026

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Colored by season — reveals the merit-order curve",
                    "Colored by year — reveals regime drift"),
    shared_yaxes=True,
    horizontal_spacing=0.07,
)

SEASON_COLORS = {"Winter": "#1f77b4", "Spring": "#2ca02c",
                 "Summer": "#ff7f0e", "Autumn": "#8c564b"}
YEAR_COLORS   = {"2023": "#d62728", "2024": "#ff7f0e",
                 "2025": "#2ca02c", "2026": "#1f77b4"}

# --- left panel: by season ---
for season, color in SEASON_COLORS.items():
    sub = scatter_df[scatter_df["season"] == season]
    fig.add_trace(
        go.Scattergl(
            x=sub["residual_load_gw"], y=sub[y_col],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.15),
            name=season, legendgroup="season",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1, col=1,
    )

# IQR band + median curve — the merit-order summary
fig.add_trace(
    go.Scatter(
        x=merit_curve["rl"], y=merit_curve["price_p75"],
        mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=merit_curve["rl"], y=merit_curve["price_p25"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(30, 30, 30, 0.15)",
        name="IQR band", legendgroup="curve", hoverinfo="skip",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=merit_curve["rl"], y=merit_curve["price_median"],
        mode="lines+markers",
        line=dict(color="black", width=3),
        marker=dict(size=5, color="black"),
        name="Median (merit-order curve)", legendgroup="curve",
        hovertemplate="Residual load: %{x:.1f} GW<br>"
                      "Median price: €%{y:.1f}/MWh<extra></extra>",
    ),
    row=1, col=1,
)

# --- right panel: by year ---
for year, color in YEAR_COLORS.items():
    sub = scatter_df[scatter_df["year"] == year]
    if len(sub) == 0:
        continue
    fig.add_trace(
        go.Scattergl(
            x=sub["residual_load_gw"], y=sub[y_col],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.2),
            name=year, legendgroup="year",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1, col=2,
    )

# Zero line on both panels — negative prices region
for c in (1, 2):
    fig.add_hline(y=0, line_dash="dot", line_color="black",
                  opacity=0.4, row=1, col=c)

fig.update_xaxes(title_text="Residual load (GW)", row=1, col=1)
fig.update_xaxes(title_text="Residual load (GW)", row=1, col=2)
fig.update_yaxes(title_text="Day-ahead price (€/MWh)", row=1, col=1)

fig.update_layout(
    title=(f"Merit-order curve: residual load vs day-ahead price (2023–2026)"
           f" — Pearson r = {pearson_r:.2f}, R² (binned median) = {r2_curve:.2f}"),
    height=560,
    template="plotly_white",
    hovermode="closest",
    legend=dict(orientation="v", yanchor="top", y=1,
                xanchor="left", x=1.02,
                groupclick="toggleitem"),
    margin=dict(l=60, r=140, t=90, b=60),
)

fig.show()

# ------------- numbers for the report -------------
print(f"Pearson correlation (linear):        r = {pearson_r:.3f}")
print(f"Spearman correlation (monotone):     r = {spearman_r:.3f}")
print(f"R² of linear fit:                    {r2_linear:.3f}")
print(f"R² of binned-median curve:           {r2_curve:.3f}")
print(f"\nResidual load range observed:        "
      f"{scatter_df['residual_load_gw'].min():.1f} to "
      f"{scatter_df['residual_load_gw'].max():.1f} GW")
print(f"Mean residual load:                  "
      f"{scatter_df['residual_load_gw'].mean():.1f} GW")
print(f"\nMerit curve — cheapest bin:          "
      f"{merit_curve['rl'].iloc[0]:.1f} GW → "
      f"€{merit_curve['price_median'].iloc[0]:.1f}/MWh")
print(f"Merit curve — most expensive bin:    "
      f"{merit_curve['rl'].iloc[-1]:.1f} GW → "
      f"€{merit_curve['price_median'].iloc[-1]:.1f}/MWh")
print(f"Merit curve — mid-range (~{merit_curve['rl'].median():.0f} GW):     "
      f"€{merit_curve['price_median'].median():.1f}/MWh")

Pearson correlation (linear):        r = 0.821
Spearman correlation (monotone):     r = 0.849
R² of linear fit:                    0.674
R² of binned-median curve:           0.695

Residual load range observed:        -3.6 to 69.0 GW
Mean residual load:                  32.0 GW

Merit curve — cheapest bin:          -0.8 GW → €-14.3/MWh
Merit curve — most expensive bin:    67.9 GW → €249.3/MWh
Merit curve — mid-range (~34 GW):     €95.1/MWh


### 4a. Price vs each generation-mix component

Same analysis, broken out by driver. Load, wind and solar each get their own panel so we can read the slope and scatter independently before combining them.

In [29]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# ------------- prep -------------
y_col = "price__germany_luxembourg"

comp = df[[
    y_col,
    "actual_cons__grid_load",
    "actual_gen__wind_onshore",
    "actual_gen__wind_offshore",
    "actual_gen__photovoltaics",
]].dropna().copy()

# Convert MWh per hour -> GW for readable axes (your hourly sums are in MWh).
comp["load_gw"]  = comp["actual_cons__grid_load"] / 1000
comp["wind_gw"]  = (comp["actual_gen__wind_onshore"]
                   + comp["actual_gen__wind_offshore"]) / 1000
comp["solar_gw"] = comp["actual_gen__photovoltaics"] / 1000

comp["hour"] = comp.index.hour
# Solar is only meaningful when it's physically possible. Including hour 02:00
# with solar = 0 GW drags the global correlation down and misrepresents solar's
# true role. A "daylight" mask picks hours where mean solar output exceeds a
# threshold, so the solar panel shows the relationship during the hours solar
# actually affects the market.
is_daylight = comp["hour"].isin(range(8, 18))  # 08:00 – 17:59 UTC (~10-19 CET)

# ------------- helper: binned median curve -------------
def binned_curve(x, y, n_bins=35, min_n=20):
    """Bin x into quantile-based bins, return median y per bin."""
    bins = pd.cut(x, bins=n_bins, labels=False, include_lowest=True)
    out = (
        pd.DataFrame({"x": x, "y": y, "bin": bins})
        .groupby("bin")
        .agg(x=("x", "mean"),
             y_med=("y", "median"),
             y_p25=("y", lambda s: s.quantile(0.25)),
             y_p75=("y", lambda s: s.quantile(0.75)),
             n=("y", "size"))
        .query("n >= @min_n")
    )
    return out

def r2_binned(x, y, n_bins=35):
    """R^2 using the binned-median curve as the predictor."""
    bins = pd.cut(x, bins=n_bins, labels=False, include_lowest=True)
    pred = pd.Series(y).groupby(bins).transform("median")
    mask = pred.notna()
    ss_res = np.sum((y[mask] - pred[mask]) ** 2)
    ss_tot = np.sum((y[mask] - y[mask].mean()) ** 2)
    return 1 - ss_res / ss_tot

# ------------- stats per component -------------
stats = []
for name, x, subset in [
    ("Load",  comp["load_gw"],                      slice(None)),
    ("Wind",  comp["wind_gw"],                      slice(None)),
    ("Solar", comp.loc[is_daylight, "solar_gw"],    "daylight only"),
]:
    if isinstance(subset, slice):
        y = comp[y_col].values
        x_vals = x.values
    else:
        y = comp.loc[is_daylight, y_col].values
        x_vals = x.values
    pr, _ = pearsonr(x_vals, y)
    sr, _ = spearmanr(x_vals, y)
    r2 = r2_binned(x_vals, y)
    stats.append(dict(
        component=name,
        subset=subset if isinstance(subset, str) else "all hours",
        n=len(y),
        pearson=pr, spearman=sr, r2=r2,
        mean_gw=x_vals.mean(),
    ))

stats_df = pd.DataFrame(stats)

# ------------- plot: 3 panels side-by-side -------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"Load vs price<br><sub>r = {stats[0]['pearson']:+.2f}, "
        f"R² = {stats[0]['r2']:.2f}</sub>",
        f"Wind vs price<br><sub>r = {stats[1]['pearson']:+.2f}, "
        f"R² = {stats[1]['r2']:.2f}</sub>",
        f"Solar vs price (daylight hours)<br><sub>r = {stats[2]['pearson']:+.2f}, "
        f"R² = {stats[2]['r2']:.2f}</sub>",
    ),
    shared_yaxes=True,
    horizontal_spacing=0.05,
)

PANEL_CONFIG = [
    ("Load",  "load_gw",  slice(None),   "#1f77b4", 1),
    ("Wind",  "wind_gw",  slice(None),   "#2ca02c", 2),
    ("Solar", "solar_gw", is_daylight,   "#ff7f0e", 3),
]

for name, col, mask, color, panel in PANEL_CONFIG:
    sub = comp if isinstance(mask, slice) else comp.loc[mask]

    # Scatter cloud
    fig.add_trace(
        go.Scattergl(
            x=sub[col], y=sub[y_col],
            mode="markers",
            marker=dict(size=2.5, color=color, opacity=0.12),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=panel,
    )

    # Binned-median curve + IQR band
    curve = binned_curve(sub[col], sub[y_col])
    fig.add_trace(
        go.Scatter(
            x=curve["x"], y=curve["y_p75"],
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=panel,
    )
    fig.add_trace(
        go.Scatter(
            x=curve["x"], y=curve["y_p25"],
            mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor="rgba(40, 40, 40, 0.15)",
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=panel,
    )
    fig.add_trace(
        go.Scatter(
            x=curve["x"], y=curve["y_med"],
            mode="lines+markers",
            line=dict(color="black", width=2.5),
            marker=dict(size=4, color="black"),
            showlegend=False,
            hovertemplate=(f"{name}: %{{x:.1f}} GW<br>"
                           f"Median price: €%{{y:.1f}}/MWh<extra></extra>"),
        ),
        row=1, col=panel,
    )

    fig.add_hline(y=0, line_dash="dot", line_color="black",
                  opacity=0.4, row=1, col=panel)

fig.update_xaxes(title_text="Load (GW)",  row=1, col=1)
fig.update_xaxes(title_text="Wind (GW)",  row=1, col=2)
fig.update_xaxes(title_text="Solar (GW)", row=1, col=3)
fig.update_yaxes(title_text="Day-ahead price (€/MWh)", row=1, col=1)

fig.update_layout(
    title="Decomposing residual load: the three components' effects on price",
    height=520,
    template="plotly_white",
    hovermode="closest",
    margin=dict(l=60, r=40, t=110, b=60),
)
fig.show()

# ------------- printed summary -------------
print("Component correlations with DE day-ahead price:\n")
print(stats_df.to_string(index=False,
    formatters={"pearson": "{:+.3f}".format,
                "spearman": "{:+.3f}".format,
                "r2": "{:.3f}".format,
                "mean_gw": "{:.1f}".format}))
print()
print(f"Reminder — residual load R² (from Section 4): 0.695")
print(f"No single component alone matches residual load, because residual")
print(f"load is their *combination* — (load - wind - solar).")

Component correlations with DE day-ahead price:

component        subset     n pearson spearman    r2 mean_gw
     Load     all hours 27743  +0.338   +0.329 0.100    53.8
     Wind     all hours 27743  -0.343   -0.357 0.115    16.1
    Solar daylight only 11560  -0.619   -0.677 0.378    15.0

Reminder — residual load R² (from Section 4): 0.695
No single component alone matches residual load, because residual
load is their *combination* — (load - wind - solar).


### 4b. Solar and wind correlations by year

A quick numeric sanity check — are the bivariate correlations stable across years, or is the relationship drifting as the generation mix changes?

In [31]:
for yr in [2023, 2024, 2025,2026]:
    yr_data = comp[comp.index.year == yr]
    daylight_yr = yr_data[yr_data["hour"].isin(range(8, 18))]
    if len(daylight_yr) < 100:
        continue
    r, _ = pearsonr(daylight_yr["solar_gw"], daylight_yr["price__germany_luxembourg"])
    print(f"{yr}: solar–price correlation (daylight) = {r:+.3f}, "
          f"n = {len(daylight_yr)}")

2023: solar–price correlation (daylight) = -0.513, n = 3650
2024: solar–price correlation (daylight) = -0.573, n = 3660
2025: solar–price correlation (daylight) = -0.724, n = 3650
2026: solar–price correlation (daylight) = -0.538, n = 600


In [33]:
for yr in [2023, 2024, 2025, 2026]:
    yr_data = comp[comp.index.year == yr]
    r, _ = pearsonr(yr_data["wind_gw"], yr_data["price__germany_luxembourg"])
    print(f"{yr}: wind–price correlation = {r:+.3f}, n = {len(yr_data)}")

2023: wind–price correlation = -0.448, n = 8760
2024: wind–price correlation = -0.336, n = 8784
2025: wind–price correlation = -0.285, n = 8760
2026: wind–price correlation = -0.530, n = 1439


In [35]:
import pandas as pd
from scipy.stats import pearsonr

SEASON_MAP = {12: "W", 1: "W", 2: "W", 3: "Sp", 4: "Sp", 5: "Sp",
              6: "Su", 7: "Su", 8: "Su", 9: "A", 10: "A", 11: "A"}
comp["season"] = comp.index.month.map(SEASON_MAP)

for yr in [2023, 2024, 2025, 2026]:
    for s in ["W", "Sp", "Su", "A"]:
        sub = comp[(comp.index.year == yr) & (comp["season"] == s)]
        if len(sub) < 200:
            continue
        r, _ = pearsonr(sub["wind_gw"], sub["price__germany_luxembourg"])
        print(f"{yr} {s}: r = {r:+.3f}")

2023 W: r = -0.732
2023 Sp: r = -0.287
2023 Su: r = -0.447
2023 A: r = -0.482
2024 W: r = -0.507
2024 Sp: r = -0.355
2024 Su: r = -0.369
2024 A: r = -0.392
2025 W: r = -0.621
2025 Sp: r = -0.143
2025 Su: r = -0.214
2025 A: r = -0.524
2026 W: r = -0.552


**Takeaway (§4).** The residual-load scatter reproduces the merit-order curve directly — the non-linear kink where gas-set pricing takes over is visible without any model. Wind has a clean negative relationship with price (more wind → lower price) that is stable year on year. Solar's correlation is strong during daylight hours but weakens in the full-day sample because zero-PV nights dilute it.

*Implication: residual load is the strongest single feature; wind is the strongest renewable driver; solar needs an hour-of-day interaction to be useful.*

## 5. Driver ranking — which signals are unique vs redundant?

**Question.** Several drivers look individually strong. Which ones carry *unique* predictive signal, and which are just reflections of the others?

This is where pairwise correlation stops being enough. Grid load, residual load, and wind are mechanically linked (residual load = grid load − renewables), so their individual correlations double-count the same information. We need three views, each answering a slightly different question:

| Method | Question it answers | Weakness |
|---|---|---|
| **Pearson \|r\|** | Is this feature individually related to price? | Double-counts shared information |
| **LMG (Shapley R²)** | What fraction of R² is uniquely attributable to this feature? | Assumes linearity |
| **Permutation importance** | How much predictive power does a trained model lose if this feature is shuffled? | Depends on the model choice |

If all three methods agree on a feature, it's a robust driver. If they disagree, the disagreement itself is the insight.

### 5a. Pairwise correlation across all drivers, month by month

In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Features to correlate against DE price. Every column here is a physical driver
# — consumption side and generation side — that plausibly affects the merit curve.
FEATURES = {
    # Consumption
    "Grid load (total)":         "actual_cons__grid_load",
    "Residual load (actual)":    "actual_residual_load",

    # Renewables
    "Wind onshore":              "actual_gen__wind_onshore",
    "Wind offshore":             "actual_gen__wind_offshore",
    "Photovoltaics":             "actual_gen__photovoltaics",
    "Biomass":                   "actual_gen__biomass",
    "Hydropower":                "actual_gen__hydropower",

    # Fossil / conventional (these are marginal-cost signals in reverse:
    # high fossil generation means the merit stack is running deep)
    "Lignite":                   "actual_gen__lignite",
    "Hard coal":                 "actual_gen__hard_coal",
    "Fossil gas":                "actual_gen__fossil_gas",
}

TARGET = "price__germany_luxembourg"
MIN_HOURS_PER_CELL = 200  # skip cells with too few observations (incomplete months)

# ---- compute monthly Pearson correlations ----
monthly = df[[TARGET, *FEATURES.values()]].copy()
monthly["year_month"] = monthly.index.to_period("M")

rows = []
for ym, group in monthly.groupby("year_month"):
    if len(group) < MIN_HOURS_PER_CELL:
        continue
    price = group[TARGET].values
    row = {"year_month": str(ym)}
    for label, col in FEATURES.items():
        x = group[col].values
        # Guard against zero-variance columns (e.g. month with all-zero solar
        # in polar latitudes — shouldn't happen for DE but be safe).
        if np.std(x) < 1e-6 or np.std(price) < 1e-6:
            row[label] = np.nan
        else:
            r, _ = pearsonr(x, price)
            row[label] = r
    rows.append(row)

corr = pd.DataFrame(rows).set_index("year_month")
# Transpose so features are rows (y-axis), months are columns (x-axis).
corr_mat = corr[list(FEATURES.keys())].T

# ---- plot ----
# Diverging colorscale centered at zero: red = positive correlation with price,
# blue = negative. Range ±0.9 — correlations above that are rare and won't
# wash out the rest of the colorscale.
fig = go.Figure(
    go.Heatmap(
        z=corr_mat.values,
        x=corr_mat.columns,
        y=corr_mat.index,
        colorscale="RdBu_r",
        zmid=0, zmin=-0.9, zmax=0.9,
        colorbar=dict(title="Pearson r", thickness=18, len=0.9),
        hovertemplate=("%{y}<br>%{x}<br>"
                       "r = %{z:.3f}<extra></extra>"),
    )
)

# Annotate each cell with the rounded correlation.
annotations = []
for i, feat in enumerate(corr_mat.index):
    for j, ym in enumerate(corr_mat.columns):
        val = corr_mat.iloc[i, j]
        if pd.isna(val):
            continue
        text_color = "white" if abs(val) > 0.55 else "black"
        annotations.append(dict(
            x=ym, y=feat, text=f"{val:+.2f}",
            showarrow=False,
            font=dict(size=8.5, color=text_color),
        ))

fig.update_layout(
    title="Month-by-month Pearson correlation with DE day-ahead price (2023–2026)",
    xaxis=dict(
        title="Month",
        tickangle=-45,
        tickmode="array",
        tickvals=list(corr_mat.columns),
        ticktext=list(corr_mat.columns),
    ),
    yaxis=dict(title=None, autorange="reversed"),
    annotations=annotations,
    height=540,
    margin=dict(l=180, r=40, t=80, b=100),
    template="plotly_white",
)

fig.show()

# ---- quick printed summary: annual means per feature ----
print("\nAnnual means of correlation (averaged across months):\n")
corr_with_year = corr.copy()
# Parse the 'YYYY-MM' strings as periods with explicit monthly freq.
corr_with_year["year"] = pd.PeriodIndex(corr_with_year.index, freq="M").year
annual = corr_with_year.groupby("year")[list(FEATURES.keys())].mean().round(2).T
print(annual.to_string())

C:\Users\ashis\AppData\Local\Temp\ipykernel_16696\4275972754.py:33: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.




Annual means of correlation (averaged across months):

year                    2023  2024  2025  2026
Grid load (total)       0.35  0.28  0.25  0.58
Residual load (actual)  0.88  0.86  0.86  0.87
Wind onshore           -0.49 -0.46 -0.37 -0.58
Wind offshore          -0.28 -0.25 -0.20 -0.19
Photovoltaics          -0.29 -0.36 -0.44 -0.17
Biomass                 0.64  0.63  0.68  0.68
Hydropower              0.40  0.55  0.45  0.23
Lignite                 0.76  0.71  0.73  0.73
Hard coal               0.62  0.56  0.65  0.72
Fossil gas              0.76  0.74  0.78  0.83


### 5b. Three-way attribution — correlation vs LMG vs permutation

In [43]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import pearsonr
from itertools import combinations

# ---- feature set ----
# For fair comparison across methods, use the same features everywhere.
# Residual load is excluded because it's a linear combo of others (would
# break LMG) — we'll discuss it separately.
FEATURES = {
    "Grid load":     "actual_cons__grid_load",
    "Wind total":    None,   # computed below
    "Solar (PV)":    "actual_gen__photovoltaics",
    "Fossil gas":    "actual_gen__fossil_gas",
    "Lignite":       "actual_gen__lignite",
    "Hard coal":     "actual_gen__hard_coal",
    "Biomass":       "actual_gen__biomass",
    "Hydropower":    "actual_gen__hydropower",
}
TARGET = "price__germany_luxembourg"

work = df.copy()
work["Wind total"] = (work["actual_gen__wind_onshore"]
                     + work["actual_gen__wind_offshore"])
FEATURES["Wind total"] = "Wind total"


# ---- method 1: Pearson correlation ----
def correlation_shares(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Signed Pearson r per feature. Absolute value used for the chart."""
    return pd.Series({c: pearsonr(X[c], y)[0] for c in X.columns})


# ---- method 2: LMG (Shapley-value decomposition of R²) ----
def lmg_shares(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Non-negative shares summing to R² of the full linear model."""
    feats = list(X.columns)
    n = len(feats)
    r2_cache = {(): 0.0}
    for size in range(1, n + 1):
        for subset in combinations(feats, size):
            key = tuple(sorted(subset))                   # <-- canonical key
            r2_cache[key] = (LinearRegression()
                             .fit(X[list(key)].values, y.values)
                             .score(X[list(key)].values, y.values))
    shares = {f: 0.0 for f in feats}
    for f in feats:
        for size in range(n):
            others = [o for o in feats if o != f]
            sub_list = list(combinations(others, size))
            marg = sum(
                r2_cache[tuple(sorted(s + (f,)))]
                - r2_cache[tuple(sorted(s))]
                for s in sub_list
            )
            shares[f] += marg / len(sub_list) / n
    return pd.Series(shares)

# ---- method 3: Permutation importance from a gradient-boosted tree ----
def model_importance(X: pd.DataFrame, y: pd.Series, random_state=0) -> pd.Series:
    """Permutation importance: mean drop in R² when a feature is shuffled."""
    model = GradientBoostingRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        random_state=random_state,
    ).fit(X.values, y.values)
    imp = permutation_importance(
        model, X.values, y.values,
        n_repeats=5, random_state=random_state, scoring="r2",
    )
    return pd.Series(imp.importances_mean, index=X.columns)


# ---- run per month ----
work["year_month"] = work.index.to_period("M")
corr_rows, lmg_rows, imp_rows = [], [], []

for ym, g in work.groupby("year_month"):
    if len(g) < 500:
        continue
    X = g[list(FEATURES.values())].copy()
    X.columns = list(FEATURES.keys())
    y = g[TARGET]
    valid = X.notna().all(axis=1) & y.notna()
    X, y = X[valid], y[valid]
    if len(X) < 500:
        continue
    key = {"year_month": str(ym)}
    corr_rows.append({**key, **correlation_shares(X, y).to_dict()})
    lmg_rows.append ({**key, **lmg_shares(X, y).to_dict()})
    imp_rows.append ({**key, **model_importance(X, y).to_dict()})

corr_df = pd.DataFrame(corr_rows).set_index("year_month")
lmg_df  = pd.DataFrame(lmg_rows).set_index("year_month")
imp_df  = pd.DataFrame(imp_rows).set_index("year_month")

feat_order = list(FEATURES.keys())

# ---- plot: three stacked charts on shared x-axis ----
# Top:    |r| from correlation (so everything is positive and comparable)
# Middle: LMG share
# Bottom: permutation importance

COLORS = {
    "Grid load":   "#c0392b",
    "Wind total":  "#2980b9",
    "Solar (PV)":  "#f39c12",
    "Fossil gas":  "#7f5539",
    "Lignite":     "#5d4037",
    "Hard coal":   "#3e2723",
    "Biomass":     "#27ae60",
    "Hydropower":  "#48c9b0",
}

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "<b>1. Correlation</b> · |Pearson r| per feature (double-counts shared information)",
        "<b>2. LMG</b> · Shapley decomposition of regression R² (true unique contribution)",
        "<b>3. Model importance</b> · Permutation importance from gradient-boosted tree (what a model actually uses)",
    ),
)

def add_stack(data, row, showlegend):
    for feat in feat_order:
        y = data[feat].abs() if row == 1 else data[feat].clip(lower=0)
        fig.add_trace(go.Bar(
            x=data.index, y=y,
            name=feat, marker_color=COLORS[feat],
            legendgroup=feat, showlegend=showlegend,
            hovertemplate=(f"<b>{feat}</b><br>%{{x}}<br>"
                           "%{y:.3f}<extra></extra>"),
        ), row=row, col=1)

add_stack(corr_df, 1, showlegend=True)
add_stack(lmg_df,  2, showlegend=False)
add_stack(imp_df,  3, showlegend=False)

fig.update_layout(
    barmode="stack",
    title=dict(
        text="<b>What drives DE day-ahead prices? Three views of the same data</b>"
             "<br><sub>Jan 2023 – Feb 2026 · each method answers a different question</sub>",
        x=0.02, xanchor="left",
    ),
    height=900, width=1400,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.05,
                xanchor="right", x=1),
    margin=dict(l=70, r=40, t=140, b=80),
)

for row, title in [(1, "Σ |r|"), (2, "Share of R²"), (3, "Δ R² on permute")]:
    fig.update_yaxes(title_text=title, row=row, col=1)
fig.update_xaxes(title_text="Month", row=3, col=1, tickangle=-45,
                 tickfont=dict(size=9))

fig.show()

# ---- printed summary: per-method annual means ----
for label, d in [("CORRELATION (|r|)", corr_df.abs()),
                 ("LMG (variance share)", lmg_df),
                 ("MODEL IMPORTANCE", imp_df)]:
    d = d.copy()
    d["year"] = pd.PeriodIndex(d.index, freq="M").year
    print(f"\n=== {label} · annual means ===")
    print(d.groupby("year")[feat_order].mean().round(3).T.to_string())

C:\Users\ashis\AppData\Local\Temp\ipykernel_16696\3230952164.py:79: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.




=== CORRELATION (|r|) · annual means ===
year         2023   2024   2025   2026
Grid load   0.354  0.283  0.263  0.579
Wind total  0.498  0.455  0.355  0.569
Solar (PV)  0.295  0.361  0.442  0.179
Fossil gas  0.761  0.739  0.780  0.831
Lignite     0.756  0.713  0.735  0.730
Hard coal   0.625  0.558  0.653  0.724
Biomass     0.636  0.631  0.679  0.679
Hydropower  0.396  0.555  0.452  0.428

=== LMG (variance share) · annual means ===
year         2023   2024   2025   2026
Grid load   0.064  0.053  0.046  0.087
Wind total  0.106  0.090  0.062  0.095
Solar (PV)  0.078  0.093  0.103  0.038
Fossil gas  0.150  0.144  0.157  0.178
Lignite     0.165  0.143  0.141  0.118
Hard coal   0.093  0.072  0.099  0.118
Biomass     0.131  0.126  0.145  0.144
Hydropower  0.043  0.087  0.061  0.059

=== MODEL IMPORTANCE · annual means ===
year         2023   2024   2025   2026
Grid load   0.017  0.021  0.013  0.006
Wind total  0.076  0.070  0.046  0.047
Solar (PV)  0.059  0.060  0.055  0.026
Fossil gas  0.

**Takeaway (§5).** Across all three methods, the ranking is consistent:

1. **Residual load** — largest unique contribution. The merit-order backbone.
2. **Wind** — consistent second. Its negative correlation with price is robust across years and methods.
3. **Grid load** — *looks* strong in pairwise correlation but shrinks dramatically under LMG. Most of its apparent signal is shared with residual load. **Don't include both.**
4. **Solar, gas, coal, biomass, hydro** — small unique contributions. Useful for marginal accuracy, not for backbone structure.

*Implication: the model needs residual load + wind as backbone, plus calendar features from §3. Adding grid load on top of residual load is a collinearity trap.*

## 6. From EDA to modelling

The EDA has told us what to build. Translating the six takeaways into a modelling plan:

### 6a. Feature engineering — what the EDA demands

| Feature group | Why it's here | Source section |
|---|---|---|
| **Calendar**: `hour`, `dow`, `is_weekend`, `month`, `season`, plus `hour × season` interaction | §3 showed two dominant cycles and a sign flip in the midday effect between winter and summer. | §3 |
| **Backbone drivers**: `residual_load`, `wind_total` (= onshore + offshore) | §5 showed these carry the unique, non-redundant signal. | §4, §5 |
| **Secondary drivers**: `solar_pv` (only via interaction with daylight hours), `fossil_gas` | §4b showed solar needs an hour gate; §5 showed gas has small but consistent unique signal as the marginal price-setter. | §4, §5 |
| **Lag features**: `price_lag_24h`, `price_lag_168h` (1 day, 1 week) | Standard for hourly load/price series — captures autocorrelation not explained by drivers. | Implied by §2 structure |
| **Regime indicator**: `year ≥ 2024` or similar | §2 showed a level shift; without this, pre-2024 data will bias the intercept. | §2 |

### 6b. Features to *exclude*

- **`grid_load`** alongside `residual_load` — §5 flagged the collinearity.
- **Any `fc_*` forecast columns as features** when the target is the realised price — they encode the same information the market already used to set the price, so they'll leak.
- **Price itself at time *t*** (obvious, but: avoid lag features shorter than the forecast horizon).

### 6c. Model choice — what the EDA suggests

The EDA reveals three things that matter for model choice:

1. **The residual-load → price relationship is non-linear** (§4, visible kink in the merit-order scatter). A pure linear model will under-fit at the high-price tail.
2. **Interactions matter** (solar × hour, hour × season). A model that handles interactions natively is preferable to hand-crafting every one.
3. **The series has regime-shifted** (§2). The model needs either a regime feature or a time-weighted training scheme.

**Recommended progression:**

| Stage | Model | Why |
|---|---|---|
| 1. Baseline | Seasonal naïve (`price[t] = price[t-168h]`) | Benchmarks how much the drivers add over pure calendar memory. |
| 2. Linear baseline | Ridge regression on engineered features | Transparent, fast, tells you where the linear signal is. |
| 3. Workhorse | **Gradient-boosted trees** (XGBoost / LightGBM) | Handles non-linearities + interactions natively. The permutation importance in §5 already used one — its ranking is directly usable. |
| 4. Sequence model | LSTM / Temporal Fusion Transformer | Only worth the complexity if (3) plateaus. The EDA here doesn't justify it yet — most of the signal is contemporaneous with the drivers, not deeply autoregressive. |

### 6d. Validation plan

- **Split by time, not randomly** — randomly shuffled splits will leak future information through the weekly seasonal cycle.
- **Rolling-origin evaluation** — train on 2023–mid-2025, test on late 2025 and Q1 2026. This stresses the regime shift flagged in §2.
- **Report MAE in €/MWh plus coverage at the negative-price tail** — the mean metric hides failure modes at the exact hours (oversupply) that are operationally most interesting.

---

### One-page summary

> **Price = f(calendar, residual load, wind) + noise.** Everything else is either a refinement or a redundant reflection of those three. The EDA supports starting with a gradient-boosted tree on ~15 engineered features, validated on a rolling time-based split, with the regime shift in 2024 handled explicitly.